# Demo 09. Conditioning and the Image Ellipse

**Standard 6** (conditioning and stability), Week 5.

A matrix maps the unit circle to an ellipse, and the 2-norm condition number equals the ellipse's elongation. This demo presents the condition number geometrically and verifies the perturbation bound.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (6, 6)

## Geometry

By the SVD $A = U\Sigma V^{\top}$, the unit circle maps to an ellipse with semi-axes equal to the singular values $\sigma_1 \ge \sigma_2$. The 2-norm condition number is their ratio

$$ \kappa_2(A) = \frac{\sigma_{\max}}{\sigma_{\min}}, $$

which governs error sensitivity through

$$ \frac{\lVert \delta x\rVert}{\lVert x\rVert} \le \kappa(A)\,\frac{\lVert \delta b\rVert}{\lVert b\rVert}. $$

A near-circular image gives $\kappa \approx 1$; a nearly degenerate ellipse gives $\kappa \gg 1$.

In [ ]:
# ---------------------------------------------------------------------------
# The image ellipse and the condition number

# ---------------------------------------------------------------------------
# The singular value decomposition A = U S V^T says: every matrix acts as
# (rotate) -> (stretch by singular values sigma_1 >= sigma_2 >= ...) -> (rotate).
# So the unit circle maps to an ellipse whose semi-axis lengths are exactly the
# singular values. The 2-norm condition number is their ratio,
#
#     kappa_2(A) = sigma_max / sigma_min,
#
# i.e. how elongated the ellipse is. kappa = 1 is a circle (perfectly
# conditioned); a long thin ellipse means a large kappa and an ill-conditioned A.

def circle_points(n=200):
    """Return points on the unit circle as a 2 x n array."""
    t = np.linspace(0, 2 * np.pi, n)
    return np.vstack([np.cos(t), np.sin(t)])

def analyse(A):
    """Return singular values and the condition number of a 2x2 matrix A."""
    U, s, Vt = np.linalg.svd(A)
    kappa = s[0] / s[1]
    return U, s, Vt, kappa

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: reshape A and watch the ellipse (and kappa) respond
# ---------------------------------------------------------------------------
def show_conditioning(a=2.0, b=1.0, c=0.0, d=1.0):
    """Draw the unit circle and its image ellipse under A = [[a,b],[c,d]].

    Overlays the principal axes (right singular vectors mapped through A) and
    prints the singular values and condition number.
    """
    A = np.array([[a, b], [c, d]])
    circ = circle_points()
    img = A @ circ                            # the ellipse

    U, s, Vt, kappa = analyse(A)

    fig, ax = plt.subplots()
    ax.plot(circ[0], circ[1], "b--", lw=1, label="unit circle")
    ax.plot(img[0], img[1], "r-", lw=2, label="image ellipse")
    # principal semi-axes: A v_i = sigma_i u_i
    for i in range(2):
        axis = A @ Vt[i]                      # = s[i] * U[:, i]
        ax.annotate("", xy=axis, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color="k", lw=2))

    lim = max(1.5, np.abs(img).max() * 1.1)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(loc="upper left")
    ax.set_title(f"sigma = ({s[0]:.3f}, {s[1]:.3f})    kappa_2(A) = {kappa:.3f}")
    plt.show()

    if kappa > 20:
        print("Large kappa: this system is ill-conditioned, small changes in the "
              "right-hand side can cause large changes in the solution.")

show_conditioning(2.0, 1.0, 0.0, 1.0)

In [ ]:
interact(
    show_conditioning,
    a=FloatSlider(value=2.0, min=-3, max=3, step=0.1, description="A[0,0]"),
    b=FloatSlider(value=1.0, min=-3, max=3, step=0.1, description="A[0,1]"),
    c=FloatSlider(value=0.0, min=-3, max=3, step=0.1, description="A[1,0]"),
    d=FloatSlider(value=1.0, min=-3, max=3, step=0.1, description="A[1,1]"),
);

In [ ]:
# ---------------------------------------------------------------------------
# The payoff: kappa bounds how much the solution can move
# ---------------------------------------------------------------------------
# Perturbation theorem:  || dx || / || x ||  <=  kappa(A) * || db || / || b ||.
# Let's confirm the bound empirically on an ill-conditioned matrix.

def perturbation_experiment(a=2.0, b=1.0, c=1.999, d=1.0, rel_noise=1e-6, trials=2000):
    """Perturb b by tiny random noise, measure the worst relative change in x,
    and compare it to the kappa * (relative change in b) bound."""
    A = np.array([[a, b], [c, d]])
    rng = np.random.default_rng(0)
    x0 = np.array([1.0, 1.0])
    b0 = A @ x0
    kappa = np.linalg.cond(A)

    worst = 0.0
    for _ in range(trials):
        db = rel_noise * np.linalg.norm(b0) * rng.standard_normal(2)
        dx = np.linalg.solve(A, b0 + db) - x0
        worst = max(worst, (np.linalg.norm(dx) / np.linalg.norm(x0)) /
                            (np.linalg.norm(db) / np.linalg.norm(b0)))
    print(f"kappa_2(A)                         = {kappa:.3e}")
    print(f"largest observed amplification     = {worst:.3e}")
    print(f"bound holds (observed <= kappa)?   {worst <= kappa * (1 + 1e-6)}")

perturbation_experiment()

## Summary

- $\kappa_2(A)=\sigma_{\max}/\sigma_{\min}$ is the elongation of the image ellipse.
- It is a property of the matrix, not of an algorithm, and bounds the sensitivity of the solution to the data.
- The perturbation experiment confirms the bound; the observed amplification does not exceed $\kappa$.
- Large $\kappa$, independent of rounding, is what makes a linear system difficult.